# episode 数据实验室 / Episode Lab

板端 `episode_recorder` 把每段演示录成 rosbag2,落在
`/home/sunrise/episodes/ep_YYYYmmdd_HHMMSS/`(含 bag、`metadata.yaml`、`meta.yaml`;
录制中/异常的目录带 `.partial` 后缀)。本笔记本在**工作站**运行(本机无 ROS),
负责把 episode 拉回本地、导出张量友好的 CSV/JPEG、画轨迹与 cmd_vel 时间线、逐帧回看并做标注。

**单向数据流 / One-way data flow**

```
板上 sunrise:~/episodes   =  采集缓存(source of truth 的原始 bag)
        │  rsync pull            ssh 灌脚本在板上用 rosbag2_py 导出
        ▼
本地   ./episodes/<ep>/     =  资产(bag + 导出的 csv/frames + annotations.yaml)
```

- **拉取只读板子**,不改板上任何东西;导出脚本在板上跑但产物写进 episode 目录随 rsync 回来。
- **标注只写本地** `annotations.yaml`,不碰 `meta.yaml`、不回写板上——板子是采集端,不是资产库。
- 切换分析对象只改一个变量 `EP = 'ep_xxxx'`,所有函数吃它。

> 录制的话题:`/scan /odom /tf /tf_static /cmd_vel /cmd_vel_joy /cmd_vel_follow`
> `/cmd_vel_mux /cmd_vel_drv /joy /safety_enabled /image_jpeg`(JPEG 压缩图)。


In [ ]:
# 配置 + ssh/rsync 工具函数:先跑这个 cell,后面所有函数都依赖它
import subprocess, json, shlex
from pathlib import Path

BOARD          = "root@192.168.13.187"          # ssh 目标(板上有 ROS2 Humble)
BOARD_EP_DIR   = "/home/sunrise/episodes"      # 板端 episode 根目录
LOCAL_EP_DIR   = Path("../episodes")              # 本地资产目录(已 .gitignore)
TROS_SETUP     = "source /opt/tros/humble/setup.bash"   # 板上 source ROS 环境
LOCAL_EP_DIR.mkdir(exist_ok=True)

EP = "ep_20260713_210340"   # ← 当前分析对象,想换哪段改这一行

def ssh(bash_cmd, timeout=60, check=False):
    """在板上跑一段 bash(已 source tros),返回 CompletedProcess。"""
    r = subprocess.run(["ssh", BOARD, bash_cmd], text=True,
                       capture_output=True, timeout=timeout)
    if check and r.returncode != 0:
        raise RuntimeError(r.stderr.strip() or f"ssh exit {r.returncode}")
    return r

def board_python(script, args=(), timeout=600):
    """把一段 Python 从 stdin 灌到板上执行(板上有 ROS),args 作为 sys.argv。
    这样脚本不用落文件、不用 scp,和 strafe_test 同一套路。"""
    argline = " ".join(shlex.quote(str(a)) for a in args)
    cmd = f"{TROS_SETUP}; export ROS_DOMAIN_ID=99; python3 /dev/stdin {argline}"
    return subprocess.run(["ssh", BOARD, cmd], input=script, text=True,
                          capture_output=True, timeout=timeout)

def local_ep(ep=None):
    """本地 episode 目录路径。"""
    return LOCAL_EP_DIR / (ep or EP)

print("ready — BOARD =", BOARD, "| EP =", EP)


In [ ]:
# 列出板上所有 episode:ssh 到板子读每个目录的 meta.yaml / rosbag2 metadata,汇总成表
LIST_SCRIPT = r"""
import os, sys, json, glob
root = sys.argv[1]
try:
    import yaml
except Exception:
    yaml = None

def load_yaml(path):
    if yaml is None or not os.path.exists(path):
        return {}
    try:
        with open(path) as f:
            return yaml.safe_load(f) or {}
    except Exception:
        return {}

def dir_size(d):
    total = 0
    for dp, _, files in os.walk(d):
        for fn in files:
            try:
                total += os.path.getsize(os.path.join(dp, fn))
            except OSError:
                pass
    return total

def find_metadata(d):
    hit = glob.glob(os.path.join(d, "**", "metadata.yaml"), recursive=True)
    return hit[0] if hit else None

rows = []
for d in sorted(glob.glob(os.path.join(root, "ep_*"))):
    if not os.path.isdir(d):
        continue
    name = os.path.basename(d)
    partial = name.endswith(".partial")
    meta = load_yaml(os.path.join(d, "meta.yaml"))
    md_path = find_metadata(d)
    bag_md = load_yaml(md_path) if md_path else {}
    # duration: 优先 meta.yaml,退回 rosbag2 metadata(纳秒)
    dur = meta.get("duration_s")
    if dur is None and bag_md:
        info = bag_md.get("rosbag2_bagfile_information", {})
        ns = info.get("duration", {}).get("nanoseconds")
        if ns is not None:
            dur = round(ns / 1e9, 1)
    rows.append({
        "name": name,
        "duration_s": dur,
        "size_mb": round(dir_size(d) / 1e6, 1),
        "partial": partial,
        "has_metadata": md_path is not None,
        "stopped_by": meta.get("stopped_by"),
        "task": meta.get("task"),
        "success": meta.get("success"),
    })
print(json.dumps(rows))
"""

def list_episodes():
    r = board_python(LIST_SCRIPT, [BOARD_EP_DIR], timeout=120)
    if r.returncode != 0:
        print(r.stderr.strip() or "ssh failed"); return None
    try:
        rows = json.loads(r.stdout.strip().splitlines()[-1])
    except (ValueError, IndexError):
        print("解析失败,板上原始输出:\n", r.stdout, r.stderr); return None
    try:
        import pandas as pd
        return pd.DataFrame(rows, columns=["name", "duration_s", "size_mb",
                            "partial", "has_metadata", "stopped_by", "task", "success"])
    except ImportError:
        for row in rows:
            print(row)
        return rows

list_episodes()


In [ ]:
# 把一个 episode 整目录 rsync 拉回本地 episodes/(只读板子,不改板上)
def pull(ep=None):
    ep = ep or EP
    LOCAL_EP_DIR.mkdir(exist_ok=True)
    src = f"{BOARD}:{BOARD_EP_DIR}/{ep}/"
    dst = local_ep(ep)
    dst.mkdir(parents=True, exist_ok=True)
    # -a 保元数据,--info=progress2 看总进度;末尾斜杠:内容对内容
    r = subprocess.run(["rsync", "-a", "--info=progress2", src, str(dst) + "/"],
                       text=True)
    if r.returncode == 0:
        print("pulled ->", dst)
    else:
        print("rsync failed, code", r.returncode)
    return dst

pull()


In [ ]:
import os
# 导出张量友好产物:调用板上唯一脚本 nav_config/episode_export.py(notebook 与 GUI 共用)
# 关键:CSV/JPEG 都是 CDR 序列化,脚本内部用 rosbag2_py + deserialize_message 解包;
#       sqlite blob 不是裸 JPEG,必须 deserialize 后取 .data。这段逻辑收敛在板上那一份脚本里。
BOARD_SCRIPT = "/home/sunrise/nav_config/episode_export.py"
LOCAL_SCRIPT = Path("../board/home/sunrise/nav_config/episode_export.py")

def _ensure_script():
    """板上没有脚本就从仓库 board/ 目录 scp 推上去(幂等)。"""
    r = ssh(f"test -f {BOARD_SCRIPT}")
    if r.returncode == 0:
        return
    print(f"板上缺 {BOARD_SCRIPT},从仓库推送 …")
    subprocess.run(["ssh", BOARD, f"mkdir -p {shlex.quote(os.path.dirname(BOARD_SCRIPT))}"],
                   check=True)
    subprocess.run(["scp", str(LOCAL_SCRIPT), f"{BOARD}:{BOARD_SCRIPT}"], check=True)

def export_csv(ep=None):
    ep = ep or EP
    _ensure_script()
    print(f"导出 {ep} …(在板上跑,大 episode 可能要几十秒)")
    cmd = f"{TROS_SETUP}; export ROS_DOMAIN_ID=99; python3 {BOARD_SCRIPT} {shlex.quote(ep)}"
    r = subprocess.run(["ssh", BOARD, cmd], text=True, capture_output=True, timeout=900)
    print(r.stdout.strip())
    if r.stderr.strip():
        print("--- stderr ---\n", r.stderr.strip())
    if "EXPORT_OK" in r.stdout:
        print("\n产物已写到板上 episode 目录,重跑 pull() 拉回本地。")

export_csv()
# pull()   # ← 导出成功后取消注释,把 csv/frames 拉回本地

In [ ]:
# 画 odom 俯视轨迹:等比例轴、绿点起点、红点终点(读本地 episodes/<EP>/odom.csv)
import csv as _csv
import matplotlib.pyplot as plt

def plot_traj(ep=None):
    ep = ep or EP
    path = local_ep(ep) / "odom.csv"
    if not path.exists():
        print("缺 odom.csv,先 export_csv() 再 pull()"); return
    xs, ys = [], []
    with open(path) as f:
        for row in _csv.DictReader(f):
            xs.append(float(row["x"])); ys.append(float(row["y"]))
    if not xs:
        print("odom.csv 为空"); return
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(xs, ys, "-", lw=1.2, color="#3b6")
    ax.plot(xs[0], ys[0], "o", color="green", ms=10, label="start")
    ax.plot(xs[-1], ys[-1], "s", color="red", ms=10, label="end")
    ax.set_aspect("equal", "datalim")
    ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
    ax.set_title(f"{ep} odom trajectory  ({len(xs)} pts)")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.show()

plot_traj()


In [ ]:
# cmd_vel 时间线:四级 vx 同图对比,看 mux 仲裁与 safety_stop 压速痕迹
def plot_cmd_vel(ep=None):
    ep = ep or EP
    d = local_ep(ep)
    # joy=源头,mux=仲裁后,drv=下发驱动,cmd_vel=最终生效;叠在一起看链路
    series = {"/cmd_vel_joy": "cmd_vel_joy.csv", "/cmd_vel_mux": "cmd_vel_mux.csv",
              "/cmd_vel_drv": "cmd_vel_drv.csv", "/cmd_vel": "cmd_vel.csv"}
    fig, ax = plt.subplots(figsize=(11, 4))
    t0 = None; plotted = False
    for label, fn in series.items():
        path = d / fn
        if not path.exists():
            continue
        ts, vx = [], []
        with open(path) as f:
            for row in _csv.DictReader(f):
                ts.append(float(row["t"])); vx.append(float(row["vx"]))
        if not ts:
            continue
        if t0 is None:
            t0 = ts[0]
        ax.step([t - t0 for t in ts], vx, where="post", label=label, alpha=0.8)
        plotted = True
    if not plotted:
        print("缺 cmd_vel csv,先 export_csv() 再 pull()"); return
    ax.set_xlabel("t [s]"); ax.set_ylabel("vx [m/s]")
    ax.set_title(f"{ep} cmd_vel vx timeline (joy → mux → drv / cmd_vel)")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.show()

plot_cmd_vel()


In [ ]:
# 逐帧回看:滑块浏览本地 frames/ 里的 JPEG(先 export_csv() + pull() 得到 frames)
from IPython.display import Image, display
import ipywidgets as widgets

def browse_frames(ep=None):
    ep = ep or EP
    fdir = local_ep(ep) / "frames"
    frames = sorted(fdir.glob("frame_*.jpg")) if fdir.exists() else []
    if not frames:
        print("没有 frames,先 export_csv() 再 pull()"); return
    out = widgets.Output()
    def show(i):
        out.clear_output(wait=True)
        with out:
            display(Image(filename=str(frames[i])))
            print(f"{i+1}/{len(frames)}  {frames[i].name}")
    slider = widgets.IntSlider(min=0, max=len(frames) - 1, step=1, value=0,
                               description="frame", continuous_update=False)
    widgets.interactive_output(show, {"i": slider})
    show(0)
    display(slider, out)

browse_frames()


In [ ]:
# 标注:写本地 episodes/<EP>/annotations.yaml(task/success/notes),不碰 meta.yaml、不回板
import yaml

def annotate(task, success, notes="", ep=None):
    ep = ep or EP
    d = local_ep(ep)
    if not d.exists():
        print("本地没有这个 episode,先 pull()"); return
    ann = {"episode": ep, "task": task, "success": bool(success), "notes": notes}
    path = d / "annotations.yaml"
    with open(path, "w") as f:
        yaml.safe_dump(ann, f, allow_unicode=True, sort_keys=False)
    print("wrote", path, "\n", ann)

def read_annotation(ep=None):
    ep = ep or EP
    path = local_ep(ep) / "annotations.yaml"
    if not path.exists():
        print("尚未标注"); return None
    with open(path) as f:
        ann = yaml.safe_load(f)
    print(ann); return ann

# 示例:annotate("pick_and_place", success=True, notes="第 3 次抓取成功,末端有轻微打滑")
read_annotation()


## 演示重放 / Replay demo(仅演示,不作验证)

ssh 触发板上 `ros2 bag play` 把录下的 cmd_vel 重新发出去,**只用于肉眼看个大概**。

> ⚠️ **为什么不能当验证 / Why this is NOT validation**
> - **走 mux 最低优先级**:重放默认 remap 到 `/cmd_vel_replay`(不是真话题),
>   即便你改发 `/cmd_vel_joy`,mux 里任何实时输入都会盖过它。
> - **初始位姿不同,轨迹必漂**:开环重放 cmd_vel,起点/地面/电量稍变,累积误差就发散。
> - **安全**:重放会让车真动——先垫起轮子或清空场地,手放急停上。

下面的 cell **默认整段注释**,看懂上面三条再手动解开。


In [ ]:
# 演示重放(默认注释):解开前先确认车已垫起/场地清空,手放急停
# def replay(ep=None, remap_to="/cmd_vel_replay"):
#     ep = ep or EP
#     # 找板上 bag(含 metadata.yaml 的目录)
#     find = (f"ls -d {BOARD_EP_DIR}/{ep} && "
#             f"find {BOARD_EP_DIR}/{ep} -name metadata.yaml")
#     md = ssh(find).stdout.strip().splitlines()
#     if len(md) < 2:
#         print('找不到 bag metadata'); return
#     bag_uri = md[-1].rsplit('/', 1)[0]
#     # 默认把 /cmd_vel remap 到 /cmd_vel_replay,绝不直接灌真话题
#     cmd = (f'{TROS_SETUP}; export ROS_DOMAIN_ID=99; '
#            f'ros2 bag play {bag_uri} --remap /cmd_vel:={remap_to}')
#     print('将在板上执行:\n', cmd)
#     # subprocess.run(['ssh', BOARD, cmd], text=True)   # ← 真要跑再解开这一行
#
# replay()
